In [12]:
import geopandas as gpd
import pandas as pd
import shapely
import copy
import os
python_directory = os.getcwd()

# Clean geogson file (remove null)

In [25]:
greens = gpd.read_file(rf"{python_directory}/port_meadow_green.geojson")
greens.fillna("")
greens.to_file(rf"{python_directory}/port_meadow_green.geojson", driver="GeoJSON")


In [15]:
paths = gpd.read_file(rf"{python_directory}/port_meadow_paths.geojson")
greens = gpd.read_file(rf"{python_directory}/port_meadow_boundary_27700.geojson")
entry_points = gpd.read_file(rf"{python_directory}/port_meadow_points.geojson")

---
# Extract nodes and links: Paths
## Example
An example for extracting the nodes:
1. Extract the x and y values from the paths geometry
2. Loop through the extracted x and y values and append the x, y, xy tuples and ids to new lists. The xy tupes are necessary to check if any points are repeated. This can occur when checking other multistrings in case of node intersections
3. Generate a pandas DataFrame listing the nodes and their coordinates

In [16]:
path = paths["geometry"][0]
print(paths["geometry"][0])

# Extract x and y coordinates of the nodes in each line in sequence
x,y = path.geoms[0].xy
x = [round(e, 5) for e in x]
y = [round(e, 5) for e in y]
xy = list(zip(x,y))
print(xy)

# Extract x and y coordinates of the entry points
xy_entry = []
for point in entry_points["geometry"]:
    x_temp, y_temp = point.xy
    xy_entry.append((x_temp[0], y_temp[0]))
print(xy_entry)

# Initialise the lists to captrure the nodes and paths properties
# nodes
xy_list = []
x_list = []
y_list = []
n_id_list = []
n_entry_list = []
# links
l_id_list = []
l_start_list = []
l_end_list = []
max_speed_list = []

n_id_counter = 0
l_id_counter = 0
# Iterate through the nodes indices
for i in range(len(x)):
    #-----------------
    # Nodes
    #-----------------
    # Check if the node has already been given an id
    if (x[i], y[i]) not in xy_list:
        # Identify entry points
        if (x[i], y[i]) in xy_entry: n_entry_list.append(1)
        else: n_entry_list.append(-1)
        # Append relevant parameters
        xy_list.append((x[i], y[i]))
        x_list.append(x[i])
        y_list.append(y[i])
        n_id_list.append(f"n_id{n_id_counter}")
        n_id_counter += 1
    #-----------------
    # Links
    #-----------------
    if i > 0:
        # Find the indices of the nodes in the full xy_lists (including previous geometries)
        i_start = xy_list.index((x[i - 1], y[i - 1]))
        i_end = xy_list.index((x[i], y[i]))
        if i_start == i_end: continue
        # Append the links
        l_start_list.append(n_id_list[i_start])
        l_end_list.append(n_id_list[i_end])
        l_id_list.append(f"l_id{l_id_counter}")
        max_speed_list.append(0)
        l_id_counter += 1

df_dict_nodes = {
        "id": n_id_list,
        "xpos": x_list,
        "ypos": y_list,
        "entry": n_entry_list
    }
df_dict_links = {
    "id": l_id_list,
    "start": l_start_list,
    "end": l_end_list
}
df_nodes = pd.DataFrame(df_dict_nodes)
df_links = pd.DataFrame(df_dict_links)
display(df_nodes)
display(df_links)


MULTILINESTRING ((448974.6973153275 209549.73660843726, 448982.3232377847 209544.78151658943, 448986.44755737705 209541.01583348334, 448999.8964256131 209529.89810240822, 449012.80733911967 209519.67696254884, 449024.6423431674 209509.81445917574, 449037.91189316026 209499.59331931637, 449046.87780531764 209492.7792260768, 449054.4091715298 209483.09604094684, 449058.6960510834 209471.9176731328))
[(448974.69732, 209549.73661), (448982.32324, 209544.78152), (448986.44756, 209541.01583), (448999.89643, 209529.8981), (449012.80734, 209519.67696), (449024.64234, 209509.81446), (449037.91189, 209499.59332), (449046.87781, 209492.77923), (449054.40917, 209483.09604), (449058.69605, 209471.91767)]
[(449100.8007841008, 209625.91316479765), (449865.5697127742, 207343.84408132924), (450222.45993505896, 207832.8085025841), (449392.62681867473, 209977.19481771026), (449393.11971142964, 209976.8528208944), (449396.46716219984, 209974.53034251207), (449398.1742370755, 209973.34595353348), (449854.8

,id,xpos,ypos,entry
0,n_id0,448974.69732,209549.73661,-1
1,n_id1,448982.32324,209544.78152,-1
2,n_id2,448986.44756,209541.01583,-1
3,n_id3,448999.89643,209529.89810,-1
4,n_id4,449012.80734,209519.67696,-1
5,n_id5,449024.64234,209509.81446,-1
6,n_id6,449037.91189,209499.59332,-1
7,n_id7,449046.87781,209492.77923,-1
8,n_id8,449054.40917,209483.09604,-1
9,n_id9,449058.69605,209471.91767,-1


,id,start,end
0,l_id0,n_id0,n_id1
1,l_id1,n_id1,n_id2
2,l_id2,n_id2,n_id3
3,l_id3,n_id3,n_id4
4,l_id4,n_id4,n_id5
5,l_id5,n_id5,n_id6
6,l_id6,n_id6,n_id7
7,l_id7,n_id7,n_id8
8,l_id8,n_id8,n_id9


In [17]:
def extract_nodes_links(paths, entry_points) -> pd.DataFrame:
    """
    Extracts the nodes from a geopandas Multistring dataframe.

    Parameters
    ----------
    - `paths`: geopandas DataFrame
        The geopandas dataframe including lines representing paths inside a green space

    Returns
    -------
    - geopandas Dataframe including the node points and their ids
    """
    # Extract x and y coordinates of the entry points
    xy_entry = []
    for point in entry_points["geometry"]:
        x_temp, y_temp = point.xy
        x_temp = [round(e, 5) for e in x_temp]
        y_temp = [round(e, 5) for e in y_temp]
        xy_entry.append((x_temp[0], y_temp[0]))
    print(xy_entry)

    # Initialise the lists to captrure the nodes and paths properties
    xy_list = []
    x_list = []
    y_list = []
    n_id_list = []
    n_sp_list = []
    n_entry_list = []
    l_id_list = []
    l_start_list = []
    l_end_list = []
    max_speed_list = []
    
    n_id_counter = 0
    l_id_counter = 0
    for path in paths["geometry"]:
        x,y = path.geoms[0].xy
        x = [round(e, 5) for e in x]
        y = [round(e, 5) for e in y]
        # Iterate through the nodes indices
        for i in range(len(x)):
            #-----------------
            # Nodes
            #-----------------
            # Check if the node has already been given an id
            if (x[i], y[i]) not in xy_list:
                xy_list.append((x[i], y[i]))
                x_list.append(x[i])
                y_list.append(y[i])
                n_id_list.append(f"n_id{n_id_counter}")
                n_sp_list.append(-1)
                n_id_counter += 1
                # Identify entry points
                if (x[i], y[i]) in xy_entry: n_entry_list.append(1)
                else: n_entry_list.append(-1)
            #-----------------
            # Links
            #-----------------
            if i > 0:
                # Find the indices of the nodes in the full xy_lists (including previous geometries)
                i_start = xy_list.index((x[i - 1], y[i - 1]))
                i_end = xy_list.index((x[i], y[i]))
                if i_start == i_end: continue
                # Append the links
                l_start_list.append(n_id_list[i_start])
                l_end_list.append(n_id_list[i_end])
                l_id_list.append(f"l_id{l_id_counter}")
                max_speed_list.append(0)
                l_id_counter += 1

    df_dict_nodes = {
        "id": n_id_list,
        "xpos": x_list,
        "ypos": y_list,
        "sp_id": n_sp_list,
        "entry": n_entry_list
    }
    df_dict_links = {
        "id": l_id_list,
        "start": l_start_list,
        "end": l_end_list,
        "maxspeed": max_speed_list
    }
    df_nodes = pd.DataFrame(df_dict_nodes)
    df_links = pd.DataFrame(df_dict_links)
    return df_nodes, df_links

## Application

In [18]:
gdf_paths = gpd.read_file(rf"{python_directory}/port_meadow_paths.geojson")
gdf_entry_points = gpd.read_file(rf"{python_directory}/port_meadow_points.geojson")

df_nodes, df_links = extract_nodes_links(gdf_paths, gdf_entry_points)

display(df_nodes)
display(df_links)
display(df_nodes.iloc[[109]])
print(xy_entry)

[(449100.80078, 209625.91316), (449865.56971, 207343.84408), (450222.45994, 207832.8085), (449392.62682, 209977.19482), (449393.11971, 209976.85282), (449396.46716, 209974.53034), (449398.17424, 209973.34595), (449854.85798, 207291.46669), (450080.3743, 206990.74571), (449569.81233, 209673.19653), (449536.65445, 209600.61965), (448599.54498, 205094.22852), (448577.63545, 205076.23226), (448537.07187, 205043.47475), (449490.91534, 205491.09577), (449503.48774, 205462.06087), (452149.37249, 204711.58942), (450476.04068, 207899.57311), (450475.22507, 207889.10561), (450474.75547, 207887.40314), (452700.17962, 207639.29113), (452607.90163, 207584.37043), (453035.49107, 206638.11465), (453035.41527, 206638.08025), (453031.73464, 206636.40888), (453030.40036, 206635.803), (453029.84207, 206635.5495), (453027.89311, 206634.66442), (453026.05684, 206633.83054), (453025.38565, 206633.52575), (453007.49458, 206630.36182), (452999.15583, 206629.91344), (452998.16975, 206629.86044), (452972.89521,

,id,xpos,ypos,sp_id,entry
0,n_id0,448974.69732,209549.73661,-1,1
1,n_id1,448982.32324,209544.78152,-1,-1
2,n_id2,448986.44756,209541.01583,-1,-1
3,n_id3,448999.89643,209529.89810,-1,-1
4,n_id4,449012.80734,209519.67696,-1,-1
...,...,...,...,...,...
4505,n_id4505,448725.32657,209431.13858,-1,-1
4506,n_id4506,448727.11976,209431.76619,-1,-1
4507,n_id4507,448729.45089,209431.76619,-1,-1
4508,n_id4508,448730.52680,209431.67653,-1,-1


,id,start,end,maxspeed
0,l_id0,n_id0,n_id1,0
1,l_id1,n_id1,n_id2,0
2,l_id2,n_id2,n_id3,0
3,l_id3,n_id3,n_id4,0
4,l_id4,n_id4,n_id5,0
...,...,...,...,...
4577,l_id4577,n_id4505,n_id4506,0
4578,l_id4578,n_id4506,n_id4507,0
4579,l_id4579,n_id4507,n_id4508,0
4580,l_id4580,n_id4508,n_id4509,0


,id,xpos,ypos,sp_id,entry
109,n_id109,449238.83798,209376.84998,-1,-1


[(449100.8007841008, 209625.91316479765), (449865.5697127742, 207343.84408132924), (450222.45993505896, 207832.8085025841), (449392.62681867473, 209977.19481771026), (449393.11971142964, 209976.8528208944), (449396.46716219984, 209974.53034251207), (449398.1742370755, 209973.34595353348), (449854.8579751864, 207291.46669264528), (450080.37429664936, 206990.74571389204), (449569.81232987053, 209673.1965252384), (449536.6544462647, 209600.61964821897), (448599.5449779812, 205094.2285169849), (448577.6354541221, 205076.23226383456), (448537.07186715305, 205043.4747456722), (449490.9153414556, 205491.09576503973), (449503.4877424249, 205462.06086719682), (452149.37249336625, 204711.58941630897), (450476.0406831457, 207899.57310590748), (450475.2250702208, 207889.10560618172), (450474.7554746713, 207887.40313995758), (452700.17961514374, 207639.2911262965), (452607.90163097053, 207584.37043189327), (453035.4910733717, 206638.11464697472), (453035.415274708, 206638.08024771954), (453031.7346

---
# Extract nodes: Polygons

In [20]:
greens = gpd.read_file(rf"{python_directory}/port_meadow_single_27700.geojson")
df_nodes_paths = df_nodes

In [21]:
green = greens["geometry"][0]
print(greens["geometry"][0])
print(green.exterior.xy)

# Extract x and y coordinates of the nodes in the outer polygon line
x,y = green.exterior.xy
xy = list(zip(x,y))
print(xy)

# Initialise the needed parameters
xy_list = []
x_list = []
y_list = []
n_id_list = []

n_id_counter = len(df_nodes_paths)
for i in range(len(x)):
    #-----------------
    # Nodes
    #-----------------
    # Check if the node has already been given an id
    if (x[i], y[i]) not in xy_list:
        xy_list.append((x[i], y[i]))
        x_list.append(x[i])
        y_list.append(y[i])
        n_id_list.append(f"n_id{n_id_counter}")
        n_id_counter += 1

df_dict_nodes_green = {
        "id": n_id_list,
        "xpos": x_list,
        "ypos": y_list
    }
df_nodes_green = pd.DataFrame(df_dict_nodes_green)
display(df_nodes_green.head())

POLYGON ((449316.4451014007 209813.73964280786, 449317.36098156613 209811.28948241024, 449317.9538749761 209811.57037336618, 449318.36617037567 209811.76036717673, 449320.46034706425 209812.74003546685, 449332.79473134025 209795.33527909662, 449345.09621925757 209779.2400985337, 449363.1938039863 209755.23732724087, 449367.4894278983 209749.28910989297, 449370.894565957 209744.14058345003, 449381.3408825296 209730.39477104013, 449392.1911901154 209715.839070984, 449410.8411660185 209690.2867497161, 449421.29569218494 209675.94118381478, 449424.445239626 209671.59245500353, 449424.78333423444 209671.22256061295, 449426.1437127401 209669.81298152677, 449430.101346758 209664.41466968268, 449430.2909389705 209662.39521092112, 449438.8738580293 209649.4591488812, 449439.3438482389 209648.98935720074, 449439.93743572605 209648.33956889313, 449440.72341909714 209647.4552848728, 449440.7413187355 209647.44168509933, 449440.85841616447 209647.27388823463, 449449.79012153926 209635.043615469, 44

,id,xpos,ypos
0,n_id4510,449316.445101,209813.739643
1,n_id4511,449317.360982,209811.289482
2,n_id4512,449317.953875,209811.570373
3,n_id4513,449318.366170,209811.760367
4,n_id4514,449320.460347,209812.740035


In [22]:
df_nodes_collated = pd.concat([df_nodes, df_nodes_green], ignore_index=True)
display(df_nodes_collated)

,id,xpos,ypos,sp_id,entry
0,n_id0,448974.697320,209549.736610,-1.0,1.0
1,n_id1,448982.323240,209544.781520,-1.0,-1.0
2,n_id2,448986.447560,209541.015830,-1.0,-1.0
3,n_id3,448999.896430,209529.898100,-1.0,-1.0
4,n_id4,449012.807340,209519.676960,-1.0,-1.0
...,...,...,...,...,...
6004,n_id6004,449285.196654,209800.143997,NaN,NaN
6005,n_id6005,449295.140044,209805.292337,NaN,NaN
6006,n_id6006,449297.992813,209806.741892,NaN,NaN
6007,n_id6007,449298.561706,209807.041783,NaN,NaN


In [23]:
df_collated = df_nodes_collated
x_max = max(df_collated["xpos"])
x_min = min(df_collated["xpos"])
y_max = max(df_collated["ypos"])
y_min = min(df_collated["ypos"])

df_nodes_normalised = copy.deepcopy(df_nodes)
df_nodes_normalised["xpos"] = [(xpos - x_min) / (x_max - x_min) for xpos in df_nodes["xpos"]]
df_nodes_normalised["ypos"] = [(ypos - y_min) / (y_max - y_min) for ypos in df_nodes["ypos"]]
display(df_nodes_normalised)
display(df_nodes_normalised[df_nodes_normalised["entry"] == 1])
print(max(df_nodes_normalised["xpos"]))

,id,xpos,ypos,sp_id,entry
0,n_id0,0.242502,0.898506,-1,1
1,n_id1,0.246946,0.896615,-1,-1
2,n_id2,0.249350,0.895178,-1,-1
3,n_id3,0.257189,0.890937,-1,-1
4,n_id4,0.264713,0.887037,-1,-1
...,...,...,...,...,...
4505,n_id4505,0.097159,0.853256,-1,-1
4506,n_id4506,0.098204,0.853495,-1,-1
4507,n_id4507,0.099563,0.853495,-1,-1
4508,n_id4508,0.100190,0.853461,-1,-1


,id,xpos,ypos,sp_id,entry
0,n_id0,0.242502,0.898506,-1,1
9,n_id9,0.291459,0.868815,-1,1
25,n_id25,0.320490,0.881508,-1,1
26,n_id26,0.315999,0.927570,-1,1
78,n_id78,0.348448,0.968135,-1,1
301,n_id301,0.103345,0.852207,-1,1
1721,n_id1721,0.543812,0.907921,-1,1
2723,n_id2723,0.969743,0.243430,-1,1
2724,n_id2724,0.939531,0.247419,-1,1
3313,n_id3313,0.761734,0.056871,-1,1


0.9941173024612256


# Save CSVs

In [24]:
df_nodes_normalised.to_csv(f"{python_directory}/node_list.csv", index=False)
df_links.to_csv(f"{python_directory}/link_list.csv", index=False)

df = df_links
display(df[df["start"] == df["end"]])

display(df_nodes_normalised[df_nodes_normalised["id"] == "n_id258"])


,id,start,end,maxspeed


,id,xpos,ypos,sp_id,entry
258,n_id258,0.093077,0.716507,-1,-1
